This experiment is dedicated to test $\hat R_{\nu}$ on different bivariate normal distribution with diagonal covariance.

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 29.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


**Package Import and other setups**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/BJX_Utility.py'
with open(utility_link) as f: exec(f.read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Hyperparam Setups**

In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

## **Experiment Part**

First Distribution with covariance matrix:
$$
\begin{bmatrix}
  1 & 0 \\
  0 & 1 \\
\end{bmatrix}
$$

In [ ]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
cov = jnp.eye(num_dim)
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1510.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005119759007357061
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0004921382642351091
Simulation Start: 1759.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0006476965500041842
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0006410024361684918
Simulation Start: 1768.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0002856976352632046
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00045708302059210837
Simulation Start: 1762.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007258480763994157
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007001494523137808
Simulation Start: 1771.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00040551135316491127
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00022953868028707802
Simulation Start: 1778.4 MB
Constrain

**Save to dataframe and file**

In [ ]:
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1_1_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1_1_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1_1_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1_1_Rhat_n.pkl"
)

Second Distribution with covariance matrix:
$$
\begin{bmatrix}
  10 & 0 \\
  0 & 1 \\
\end{bmatrix}
$$

In [ ]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
cov = jnp.array([
    [10.0, 0.0],
    [0.0, 1.0]
])
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
ten_MSE_c_list = []
ten_MSE_n_list = []
ten_RHat_c_list = []
ten_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, ten_RHat_c_list,ten_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, ten_RHat_n_list,ten_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1511.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0007269943598657846
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0007371189421974123
Simulation Start: 1779.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.000544836453627795
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00037008716026321054
Simulation Start: 1775.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00048591833910904825
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00050029979320243
Simulation Start: 1777.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005819338839501143
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00048748048720881343
Simulation Start: 1776.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0002374340547248721
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00021672071306966245
Simulation Start: 1779.3 MB
Constrained

**Save to dataframe and file**

In [ ]:
MSE_c_df = pd.DataFrame(ten_MSE_c_list)
R_Hat_c_df = pd.DataFrame(ten_RHat_c_list)

MSE_n_df = pd.DataFrame(ten_MSE_n_list)
R_Hat_n_df = pd.DataFrame(ten_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_10_1_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_10_1_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_10_1_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_10_1_Rhat_n.pkl"
)

Third Distribution with covariance matrix:
$$
\begin{bmatrix}
  100 & 0 \\
  0 & 1 \\
\end{bmatrix}
$$

In [ ]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
cov = jnp.array([
    [100.0, 0.0],
    [0.0, 1.0]
])
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
hun_MSE_c_list = []
hun_MSE_n_list = []
hun_RHat_c_list = []
hun_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, hun_RHat_c_list,hun_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, hun_RHat_n_list,hun_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1511.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0007053966401144862
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00045641427277587354
Simulation Start: 1761.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00036519108107313514
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0002423879923298955
Simulation Start: 1763.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000466386612970382
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00045982105075381696
Simulation Start: 1763.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0006791841005906463
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006969106616452336
Simulation Start: 1779.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006856434047222137
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005696540465578437
Simulation Start: 1787.7 MB
Constraine

**Save to dataframe and file**

In [ ]:
MSE_c_df = pd.DataFrame(hun_MSE_c_list)
R_Hat_c_df = pd.DataFrame(hun_RHat_c_list)

MSE_n_df = pd.DataFrame(hun_MSE_n_list)
R_Hat_n_df = pd.DataFrame(hun_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_100_1_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_100_1_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_100_1_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_100_1_Rhat_n.pkl"
)

4-th Distribution with covariance matrix:
$$
\begin{bmatrix}
  1000 & 0 \\
  0 & 1 \\
\end{bmatrix}
$$

In [ ]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
cov = jnp.array([
    [1000.0, 0.0],
    [0.0, 1.0]
])
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
thou_MSE_c_list = []
thou_MSE_n_list = []
thou_RHat_c_list = []
thou_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, thou_RHat_c_list,thou_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, thou_RHat_n_list,thou_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1526.2 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0003680869413074106
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0003180574276484549
Simulation Start: 1778.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00018302984244655818
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00023703253827989101
Simulation Start: 1775.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00020998665422666818
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00036216567968949676
Simulation Start: 1783.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00031043775379657745
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00014402097440324724
Simulation Start: 1783.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0002343954984098673
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0001682057190919295
Simulation Start: 1792.7 MB
Constr

In [ ]:
MSE_c_df = pd.DataFrame(thou_MSE_c_list)
R_Hat_c_df = pd.DataFrame(thou_RHat_c_list)

MSE_n_df = pd.DataFrame(thou_MSE_n_list)
R_Hat_n_df = pd.DataFrame(thou_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1000_1_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1000_1_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1000_1_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Diag_1000_1_Rhat_n.pkl"
)